# RL Experiment 05: Reward Shaping

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_context()`, `create_env()` |
| `src/schedule_engine/rl/gym_env/` | Reward calculator | `RewardCalculator` |
| **This notebook** | Experiment-specific config | Reward comparison |

## Experiment Overview
- **Goal**: Compare scalar vs hypervolume reward calculations
- **Variants**: Scalar fitness delta vs multi-objective hypervolume contribution
- **Metrics**: Reward components and magnitudes across transitions

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    load_context,
    set_global_seed,
)
from schedule_engine.rl.gym_env.reward_calculator import RewardCalculator

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:
# ============================================================================
# RL EXPERIMENT 05 CONFIGURATION - Reward Shaping Comparison
# ============================================================================

SEED = 42
POP_SIZE = 20
MAX_GENERATIONS = 30
MAX_STEPS = 10
NUM_TRANSITIONS = 10  # Number of transitions to sample for comparison

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_05_reward_shaping_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: pop={POP_SIZE}, ngen={MAX_GENERATIONS}, transitions={NUM_TRANSITIONS}")
print(f" Output: {OUTPUT_DIR}")

## 3. Load Data & Create Environment

In [ ]:
# Set reproducibility
set_global_seed(SEED)

# Build config and load scheduling context
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context(DATA_DIR, config)

# Create RL environment
env = create_env(
    context=context,
    pop_size=POP_SIZE,
    max_generations=MAX_GENERATIONS,
    max_steps=MAX_STEPS,
)

print(f" Environment created with population of {len(env.population)} individuals")

## 4. Compare Reward Calculation Methods

In [ ]:
# Create both reward calculators
scalar_calc = RewardCalculator(use_hypervolume=False)
hv_calc = RewardCalculator(use_hypervolume=True)

# Sample transitions from population
population = env.population
comparison_results = []

for i in range(min(NUM_TRANSITIONS, len(population) - 1)):
    prev_ind = population[i]
    new_ind = population[i + 1]
    
    # Calculate rewards using both methods
    scalar_reward, scalar_components = scalar_calc.calculate_reward(
        prev_individual=prev_ind,
        new_individual=new_ind,
        population_diversity=0.1,
        generation=i,
        population=population,
    )
    
    hv_reward, hv_components = hv_calc.calculate_reward(
        prev_individual=prev_ind,
        new_individual=new_ind,
        population_diversity=0.1,
        generation=i,
        population=population,
    )
    
    comparison_results.append({
        "transition": i,
        "scalar_reward": scalar_reward,
        "scalar_components": scalar_components,
        "hv_reward": hv_reward,
        "hv_components": hv_components,
    })
    
    print(f"Transition {i}: Scalar={scalar_reward:.4f}, Hypervolume={hv_reward:.4f}")

## 5. Results Summary

In [ ]:
import numpy as np

# Calculate summary statistics
scalar_rewards = [r["scalar_reward"] for r in comparison_results]
hv_rewards = [r["hv_reward"] for r in comparison_results]

print(f"\n{'='*60}")
print(f"RL EXPERIMENT 05: REWARD SHAPING COMPARISON RESULTS")
print(f"{'='*60}")
print(f"\nScalar Reward Statistics:")
print(f"  Mean: {np.mean(scalar_rewards):.4f}")
print(f"  Std:  {np.std(scalar_rewards):.4f}")
print(f"  Min:  {np.min(scalar_rewards):.4f}")
print(f"  Max:  {np.max(scalar_rewards):.4f}")

print(f"\nHypervolume Reward Statistics:")
print(f"  Mean: {np.mean(hv_rewards):.4f}")
print(f"  Std:  {np.std(hv_rewards):.4f}")
print(f"  Min:  {np.min(hv_rewards):.4f}")
print(f"  Max:  {np.max(hv_rewards):.4f}")

# Correlation between methods
correlation = np.corrcoef(scalar_rewards, hv_rewards)[0, 1]
print(f"\nCorrelation between methods: {correlation:.4f}")
print(f"{'='*60}")

## 6. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_05_reward_shaping",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "max_generations": MAX_GENERATIONS,
        "max_steps": MAX_STEPS,
        "num_transitions": NUM_TRANSITIONS,
    },
    "results": {
        "scalar": {
            "mean": float(np.mean(scalar_rewards)),
            "std": float(np.std(scalar_rewards)),
            "min": float(np.min(scalar_rewards)),
            "max": float(np.max(scalar_rewards)),
        },
        "hypervolume": {
            "mean": float(np.mean(hv_rewards)),
            "std": float(np.std(hv_rewards)),
            "min": float(np.min(hv_rewards)),
            "max": float(np.max(hv_rewards)),
        },
        "correlation": float(correlation),
        "transition_details": comparison_results,
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2, default=str)

print(f" Results saved to: {results_path}")